In [0]:
for table in [
    "bronze_cpi_monthly", "bronze_hh_income_national", 
    "bronze_hies_state", "bronze_hies_percentile"
]:
    print(f"\n{'='*50}")
    print(f"my_living_index.bronze.{table}")
    print('='*50)
    sdf = spark.table(f"my_living_index.bronze.{table}")
    sdf.printSchema()
    display(sdf.limit(3))

In [0]:
spark.table("my_living_index.bronze.bronze_hies_percentile") \
    .select("variable").distinct().show()

In [0]:
CATALOG = "my_living_index"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# CPI Monthly
print("Processing silver_cpi_monthly...")

cpi_raw = spark.table(f"{CATALOG}.bronze.bronze_cpi_monthly")

KEY_DIVISIONS = ["overall", "01", "04", "07", "12"]

silver_cpi = (
    cpi_raw
    .filter(F.col("division").isin(KEY_DIVISIONS))
    .withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("division_label",
                F.when(F.col("division") == "overall", "Overall CPI")
                .when(F.col("division") == "01", "Food & Non-Alcoholic Beverages")
                .when(F.col("division") == "04", "Housing, Water & Utilities")
                .when(F.col("division") == "07", "Transport")
                .when(F.col("division") == "12", "Misc (Health, Education)")
                .otherwise(F.col("division"))
    )
    .withColumn("index", F.col("index").cast("double"))
    .withColumn("index_lag_12m",
                F.lag("index", 12).over(
                    Window.partitionBy("division").orderBy("date")
                )
    )
    .withColumn("yoy_infaltion_pct",
                F.round(
                    ((F.col("index") - F.col("index_lag_12m")) / F.col("index_lag_12m")) * 100, 2
                )
    )
    .drop("index_lag_12m", "_source_url", "_ingested_at", "_source_row_count")
    .orderBy("date", "division")
)

silver_cpi.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.silver_cpi_monthly")

print(f"silver_cpi_monthly: {silver_cpi.count():,} rows")


# Household Income National
print("\nProcessing silver_hh_income_national...")

hh_raw = spark.table(f"{CATALOG}.bronze.bronze_hh_income_national")

silver_hh = (
    hh_raw
    .withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
    .withColumn("year", F.year("date"))
    .withColumn("income_mean", F.col("income_mean").cast("double"))
    .withColumn("income_median", F.col("income_median").cast("double"))
    .withColumn("geography", F.lit("Malaysia"))
    .withColumn("mean_lag",
                F.lag("income_mean", 1).over(Window.orderBy("date"))
    )
    .withColumn("median_lag",
                F.lag("income_median", 1).over(Window.orderBy("date"))
    )
    .withColumn("mean_growth_pct",
                F.round(
                    ((F.col("income_mean") - F.col("mean_lag")) / F.col("mean_lag")) * 100, 2
                )
    )
    .withColumn("median_growth_pct",
                F.round(
                    ((F.col("income_median") - F.col("median_lag")) / F.col("median_lag")) * 100, 2
                )
    )
    .drop("mean_lag", "median_lag", "_source_url", "_ingested_at", "_source_row_count")
    .orderBy("date")
)

silver_hh.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.silver_hh_income_national")

print(f"silver_hh_income_national: {silver_hh.count():,} rows")


# HIES State
print("\nProcessing silver_hies_state")

hies_raw = spark.table(f"{CATALOG}.bronze.bronze_hies_state")

silver_hies = (
    hies_raw
    .withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
    .withColumn("year", F.year("date"))
    .withColumn("income_mean", F.col("income_mean").cast("double"))
    .withColumn("income_median", F.col("income_median").cast("double"))
    .withColumn("expenditure_mean", F.col("expenditure_mean").cast("double"))
    .withColumn("gini", F.col("gini").cast("double"))
    .withColumn("poverty", F.col("poverty").cast("double"))
    .withColumn("expenditure_to_income_ratio",
                F.round(F.col("expenditure_mean") / F.col("income_mean"), 4))
    .withColumn("monthly_surplus",
                F.round(F.col("income_mean") - F.col("expenditure_mean"), 2))
    # Rank states: rank 1 = lowest expenditure ratio
    .withColumn("affordability_rank",
                F.rank().over(Window.partitionBy("year").orderBy("expenditure_to_income_ratio")))
    .drop("_source_url", "_ingested_at", "_source_row_count")
    .orderBy("year", "affordability_rank")
)

silver_hies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.silver_hies_state")

print(f"silver_hies_state: {silver_hies.count():,} rows")


# Income Percentile
print("\nProcessing silver_income_percentile...")

pct_raw = spark.table(f"{CATALOG}.bronze.bronze_hies_percentile")

silver_pct = (
    pct_raw
    .withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
    .withColumn("year", F.year("date"))
    .withColumn("income", F.col("income").cast("double"))
    .withColumn("percentile", F.col("percentile").cast("integer"))
    # Pivot variable into columns
    .groupBy("date", "year", "percentile")
    .pivot("variable", ["mean", "median", "minimum", "maximum"])
    .agg(F.first("income"))
    # Rename columns
    .withColumnRenamed("mean", "income_mean")
    .withColumnRenamed("median", "income_median")
    .withColumnRenamed("minimum", "income_minimum")
    .withColumnRenamed("maximum", "income_maximum")
    # Assign salary bracket group
    .withColumn("income_group",
                F.when(F.col("percentile") <= 40, "B40")
                 .when(F.col("percentile") <= 80, "M40")
                 .otherwise("T20"))
    # Income range width per percentile bucket
    .withColumn("income_range",
                F.round(F.col("income_maximum") - F.col("income_minimum"), 2))
    .orderBy("date", "percentile")
)

silver_pct.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.silver_income_percentile")

print(f"silver_income_percentile: {silver_pct.count():,} rows")